In [1]:
import os

os.chdir("../")

In [2]:
from llm import ask_groq
from IPython.display import Markdown, display

# Example usage (teaching-friendly)
question = "Who won the WB election?"
answer = ask_groq(question)

# Nicely formatted output for notebook display
display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

**Question:** Who won the WB election?

**Answer:** I'm not aware of any recent information about a "WB election". Could you please provide more context or clarify which election you are referring to? I'll do my best to provide you with the most accurate and up-to-date information.

In [3]:
wikipedia_context = """
Legislative Assembly elections were held in West Bengal to elect all 294 members of the West Bengal Legislative Assembly in two phases on 23 and 29 April 2026,[4] with the votes counted and results for 293 seats released on 4 May 2026.

Over 9 million voters were removed through the Special Intensive Revision (SIR) prior to the elections. This move has been criticized as erosion of democracy in India.[5][6]

The election saw a historic defeat for the incumbent All India Trinamool Congress which was ruling since 2011, with the Bharatiya Janata Party becoming the first right-wing party to be elected in the state since assembly elections first began in 1937. With a voter turnout of 92.93%, the election was the most widely participated in West Bengal, surpassing the 2011 election.

In a move unprecedented in Indian politics, Mamata Banerjee, the incumbent Trinamool Congress chief minister, refused to resign her office despite losing her seat and a majority in the assembly as a result of the election, alleging irregularities in its conduct.[7][8][9] Her tenure as chief minister came to an end after the dissolution of the assembly by the governor at the end of its term on 7 May 2026.
"""

question = "Who won the WB election?"
question_with_context = f"{wikipedia_context}\n\nQuestion: {question}"
answer = ask_groq(question_with_context)

# Nicely formatted output for notebook display
display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

**Question:** Who won the WB election?

**Answer:** The Bharatiya Janata Party (BJP) won the West Bengal Legislative Assembly election, becoming the first right-wing party to be elected in the state since assembly elections first began in 1937, thereby defeating the incumbent All India Trinamool Congress.

In [4]:
from pydantic import BaseModel, Field
from typing import Any

class ToolUsage(BaseModel):
    tool: str = Field(description="Exact name of the tool to use")
    args: dict[str, Any] = Field(description="Arguments to pass to the tool")
    reason: str = Field(description="Reason for using the tool")


schema = ToolUsage.model_json_schema()
schema


{'properties': {'tool': {'description': 'Exact name of the tool to use',
   'title': 'Tool',
   'type': 'string'},
  'args': {'additionalProperties': True,
   'description': 'Arguments to pass to the tool',
   'title': 'Args',
   'type': 'object'},
  'reason': {'description': 'Reason for using the tool',
   'title': 'Reason',
   'type': 'string'}},
 'required': ['tool', 'args', 'reason'],
 'title': 'ToolUsage',
 'type': 'object'}

In [13]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

# MCP server endpoint — override via the MARKETINTEL_ENDPOINT environment variable.
SERVER_ENDPOINT = "http://127.0.0.1:8000/mcp"
transport = StreamableHttpTransport(url=SERVER_ENDPOINT)
client = Client(transport)

In [15]:
async with client:
    tools = await client.list_tools()
    tools = [tool.model_dump() for tool in tools]
    prompt_response = await client.get_prompt("tool_selection_prompt", arguments={"query": question, "tools": tools, "schema": schema})
    prompt = prompt_response.messages[0].content.text
    response = ask_groq(question=prompt)
    print(response)

[{"tool": "web_search", "args": {"query": "WB election results"}, "reason": "To find the latest information on the WB election results, a web search is necessary as it can provide up-to-date news and announcements from reliable sources."}]


In [16]:
import re
import json

def get_json(response: str) -> list[ToolUsage]:
    json_str = re.search(r"\[.*\]", response, re.DOTALL).group(0)
    return json.loads(json_str)

tool_usages = get_json(response)
tool_usages

[{'tool': 'web_search',
  'args': {'query': 'WB election results'},
  'reason': 'To find the latest information on the WB election results, a web search is necessary as it can provide up-to-date news and announcements from reliable sources.'}]

In [17]:
async with client:
    for tool_usage in tool_usages:
        print(f"Calling tool: {tool_usage['tool']} with arguments: {tool_usage['args']} because {tool_usage['reason']}")
        tool_response = await client.call_tool(tool_usage["tool"], arguments=tool_usage["args"])
        print(f"Response from tool {tool_usage['tool']}: \n{tool_response.content[0].text}")

Calling tool: web_search with arguments: {'query': 'WB election results'} because To find the latest information on the WB election results, a web search is necessary as it can provide up-to-date news and announcements from reliable sources.
Response from tool web_search: 
{'query': 'WB election results', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://westbengalelection.com/results', 'title': 'Live Results - West Bengal Election 2026', 'content': '[WB WB Election West Bengal Opinion Polls 2026](https://westbengalelection.com/). *   [Parties](https://westbengalelection.com/parties-alliances). *   [Constituencies](https://westbengalelection.com/constituencies). *   [LIVE VOTING](https://westbengalelection.com/vote). *   [Parties](https://westbengalelection.com/parties-alliances). *   [Constituencies](https://westbengalelection.com/constituencies). [LIVE VOTING](https://westbengalelection.com/vote). [View Exit Polls](https://westbengalelection.com/

In [18]:
import ast

tool_response_data = ast.literal_eval(tool_response.content[0].text)
tool_response_data

{'query': 'WB election results',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://westbengalelection.com/results',
   'title': 'Live Results - West Bengal Election 2026',
   'content': '[WB WB Election West Bengal Opinion Polls 2026](https://westbengalelection.com/). *   [Parties](https://westbengalelection.com/parties-alliances). *   [Constituencies](https://westbengalelection.com/constituencies). *   [LIVE VOTING](https://westbengalelection.com/vote). *   [Parties](https://westbengalelection.com/parties-alliances). *   [Constituencies](https://westbengalelection.com/constituencies). [LIVE VOTING](https://westbengalelection.com/vote). [View Exit Polls](https://westbengalelection.com/results#exit-polls). ### ![Image 1](https://westbengalelection.com/images/icon/bjp.webp) BJP. ### ![Image 2](https://westbengalelection.com/images/icon/tmc.webp) AITC. ### ![Image 3](https://westbengalelection.com/images/icon/inc.webp) INC. ### ![Image 4](https://

In [27]:

query = question
search_results = tool_response_data

async with client:
    formatting_instruction_response = await client.read_resource("resource://response_formatting_instructions")
    formatting_instructions = formatting_instruction_response[0].text
    prompt_response = await client.get_prompt("main_prompt", arguments={"query": query, "search_results": search_results, "formatting_instructions": formatting_instructions})
    prompt = prompt_response.messages[0].content.text
    analysis_response = ask_groq(question=prompt)
    print(analysis_response)
# formatting_instruction_response

## Introduction to West Bengal Election Results
The West Bengal election results are out, and the winning party has been announced. 

## Winner of the WB Election
The **Bharatiya Janata Party (BJP)** has won the West Bengal election with **207 seats**. 

## Party-Wise Results
Here is a summary of the party-wise results:
* **Bharatiya Janata Party (BJP)**: 207 seats
* **All India Trinamool Congress (AITC)**: 80 seats
* **Indian National Congress (INC)**: 2 seats
* **Aam Janata Unnayan party (AJUP)**: 2 seats
* **Communist Party of India (Marxist) (CPI(M))**: 1 seat
* **All India Secular Front (AISF)**: 1 seat

## Sources
For more information, you can visit the following websites:
* [West Bengal Election Results](https://results.eci.gov.in/ResultAcGenMay2026/partywiseresult-S25.htm)
* [CEO West Bengal](https://ceowestbengal.wb.gov.in/Election)
* [The Times of India](https://timesofindia.indiatimes.com/elections/assembly-elections/west-bengal/results)

## Conclusion
The **BJP** has emerge

In [28]:
from IPython.display import Markdown
Markdown(analysis_response)

## Introduction to West Bengal Election Results
The West Bengal election results are out, and the winning party has been announced. 

## Winner of the WB Election
The **Bharatiya Janata Party (BJP)** has won the West Bengal election with **207 seats**. 

## Party-Wise Results
Here is a summary of the party-wise results:
* **Bharatiya Janata Party (BJP)**: 207 seats
* **All India Trinamool Congress (AITC)**: 80 seats
* **Indian National Congress (INC)**: 2 seats
* **Aam Janata Unnayan party (AJUP)**: 2 seats
* **Communist Party of India (Marxist) (CPI(M))**: 1 seat
* **All India Secular Front (AISF)**: 1 seat

## Sources
For more information, you can visit the following websites:
* [West Bengal Election Results](https://results.eci.gov.in/ResultAcGenMay2026/partywiseresult-S25.htm)
* [CEO West Bengal](https://ceowestbengal.wb.gov.in/Election)
* [The Times of India](https://timesofindia.indiatimes.com/elections/assembly-elections/west-bengal/results)

## Conclusion
The **BJP** has emerged as the winner of the West Bengal election, with **207 seats**. You can find more detailed information on the election results on the above-mentioned websites.